In [10]:
import pandas as pd
import numpy as np
import random
import json
import time
import folium
import os
from folium import plugins

In [11]:
# FUNCTION DEFINITIONS
def hitung_total_jarak(rute, matriks):
    """Menghitung total jarak dari rute (0 -> rute -> 0)"""
    jarak = 0
    # Dari Depot ke titik pertama
    jarak += matriks.iloc[0, rute[0]]
    # Antar titik di dalam rute
    for i in range(len(rute) - 1):
        jarak += matriks.iloc[rute[i], rute[i+1]]
    # Kembali ke Depot dari titik terakhir
    jarak += matriks.iloc[rute[-1], 0]
    return jarak

def buat_populasi_awal(ukuran_populasi, jumlah_titik):
    """Membuat rute acak (tidak termasuk index 0/depot)"""
    populasi = []
    titik_puskesmas = list(range(1, jumlah_titik)) # Index 1 s/d N-1
    for _ in range(ukuran_populasi):
        rute = random.sample(titik_puskesmas, len(titik_puskesmas))
        populasi.append(rute)
    return populasi

def seleksi_tournament(populasi, matriks, k=3):
    """Memilih induk terbaik dari k rute acak"""
    tournament = random.sample(populasi, k)
    tournament.sort(key=lambda r: hitung_total_jarak(r, matriks))
    return tournament[0]

def ordered_crossover(induk1, induk2):
    """Mengawinkan 2 rute tanpa ada titik yang kembar (OX)"""
    size = len(induk1)
    a, b = sorted(random.sample(range(size), 2))
    
    anak = [None] * size
    anak[a:b] = induk1[a:b]
    
    # Isi sisanya dari induk 2
    pos_anak = b
    for titik in induk2:
        if titik not in anak:
            if pos_anak >= size:
                pos_anak = 0
            while anak[pos_anak] is not None:
                pos_anak = (pos_anak + 1) % size
            anak[pos_anak] = titik
    return anak

def swap_mutation(rute, laju_mutasi):
    """Menukar dua titik secara acak jika memenuhi probabilitas"""
    if random.random() < laju_mutasi:
        idx1, idx2 = random.sample(range(len(rute)), 2)
        rute[idx1], rute[idx2] = rute[idx2], rute[idx1]
    return rute

In [12]:
# MAIN FUNCTION
def jalankan_ga(file_matriks, n_populasi=100, n_generasi=500, pc=0.8, pm=0.2):
    df_matriks = pd.read_csv(file_matriks, index_col=0)
    jumlah_titik = len(df_matriks)
    
    # 1. Inisialisasi
    populasi = buat_populasi_awal(n_populasi, jumlah_titik)
    best_ever = min(populasi, key=lambda r: hitung_total_jarak(r, df_matriks))
    
    # === WADAH UNTUK GRAFIK KONVERGENSI ===
    riwayat_jarak = [] 
    
    # 2. Evolusi dengan Elitism
    for gen in range(n_generasi):
        # Elitism: Simpan dulu best_ever agar tidak hilang
        populasi_baru = [best_ever] 
        
        # Isi sisa populasi sampai penuh (n_populasi - 1)
        while len(populasi_baru) < n_populasi:
            p1 = seleksi_tournament(populasi, df_matriks)
            p2 = seleksi_tournament(populasi, df_matriks)
            
            # Crossover
            if random.random() < pc:
                anak = ordered_crossover(p1, p2)
            else:
                anak = p1.copy()
            
            # Mutasi
            anak = swap_mutation(anak, pm)
            populasi_baru.append(anak)
            
        populasi = populasi_baru
        current_best = min(populasi, key=lambda r: hitung_total_jarak(r, df_matriks))
        
        # Update best_ever
        if hitung_total_jarak(current_best, df_matriks) < hitung_total_jarak(best_ever, df_matriks):
            best_ever = current_best
            
        # === CATAT JARAK TERBAIK DI GENERASI INI ===
        riwayat_jarak.append(hitung_total_jarak(best_ever, df_matriks))
            
    # Hasil akhir (tambahkan index 0 di awal dan akhir)
    rute_final = [0] + best_ever + [0]
    jarak_final = hitung_total_jarak(best_ever, df_matriks)
    
    # Kembalikan riwayat_jarak sebagai output tambahan
    return rute_final, jarak_final, list(df_matriks.columns), riwayat_jarak

In [13]:
# MAIN EXECUTION
klaster_list = ['Barat', 'Pusat', 'Selatan', 'Timur', 'Utara']

# Bikin 'brankas' untuk menyimpan hasil GA 
hasil_semua_klaster = {} 
total_jarak_seluruh_sby = 0

print("🚀 MEMULAI PENCARIAN RUTE (GA) 🚀\n" + "="*50)

waktu_mulai_total = time.time()

for klaster in klaster_list:
    file_matriks = f"matriks_jarak_{klaster}.csv"
    
    waktu_mulai_klaster = time.time()
    
    # Jalankan mesin GA! (Tambahkan variabel penangkap 'riwayat')
    rute_indeks, jarak, nama_lokasi, riwayat = jalankan_ga(file_matriks)
    
    waktu_selesai_klaster = time.time()
    runtime_klaster = waktu_selesai_klaster - waktu_mulai_klaster
    
    # Terjemahkan nomor indeks rute (0, 3, 1...) menjadi nama Puskesmas asli
    rute_nama = [nama_lokasi[i] for i in rute_indeks]
    
    # Simpan ke dalam brankas (Tambahkan key 'riwayat_konvergensi')
    hasil_semua_klaster[klaster] = {
        'rute_indeks': rute_indeks,
        'rute_nama': rute_nama,
        'jarak': jarak,
        'runtime': runtime_klaster,
        'riwayat_konvergensi': riwayat
    }
    total_jarak_seluruh_sby += jarak
    
    print(f"📍 KLASTER {klaster.upper()}")
    print(f"Jarak Tempuh : {jarak:.3f} KM")
    print(f"Waktu Komputasi : {runtime_klaster:.3f} detik") 
    print(f"Urutan Rute  : {' ➔ '.join(rute_nama)}")
    print("-" * 50)

waktu_selesai_total = time.time()
runtime_total = waktu_selesai_total - waktu_mulai_total

print(f"🏆 TOTAL KESELURUHAN JARAK SURABAYA: {total_jarak_seluruh_sby:.3f} KM")
print(f"⏱️ TOTAL WAKTU EKSEKUSI (5 KLASTER) : {runtime_total:.3f} detik")

🚀 MEMULAI PENCARIAN RUTE (GA) 🚀
📍 KLASTER BARAT
Jarak Tempuh : 44.983 KM
Waktu Komputasi : 40.234 detik
Urutan Rute  : UPTD Gudang Farmasi Surabaya ➔ Puskesmas Simomulyo Surabaya ➔ Puskesmas Asemrowo Surabaya ➔ Puskesmas Tanjungsari Surabaya ➔ Puskesmas Balongsari Surabaya ➔ Puskesmas Manukan Kulon Surabaya ➔ Puskesmas Sememi Surabaya ➔ Puskesmas Benowo Surabaya ➔ Puskesmas Made Surabaya ➔ Puskesmas Lontar Surabaya ➔ Puskesmas Lidah Kulon Surabaya ➔ Puskesmas Jeruk Surabaya ➔ Puskesmas Bangkingan Surabaya ➔ UPTD Gudang Farmasi Surabaya
--------------------------------------------------
📍 KLASTER PUSAT
Jarak Tempuh : 26.111 KM
Waktu Komputasi : 27.687 detik
Urutan Rute  : UPTD Gudang Farmasi Surabaya ➔ Puskesmas Dr. Soetomo Surabaya ➔ Puskesmas Kedungdoro Surabaya ➔ Puskesmas Tembok Dukuh Surabaya ➔ Puskesmas Gundih Surabaya ➔ Puskesmas Peneleh Surabaya ➔ Puskesmas Simolawang Surabaya ➔ Puskesmas Tambakrejo Surabaya ➔ Puskesmas Ketabang Surabaya ➔ UPTD Gudang Farmasi Surabaya
----------

In [14]:
def cek_constraint_rute(nama_klaster, rute_indeks, jumlah_titik_seharusnya):
    """
    Fungsi untuk mengecek apakah rute hasil GA melanggar aturan TSP.
    """
    print(f"🔍 MEMERIKSA CONSTRAINT KLASTER: {nama_klaster.upper()}")
    print(f"Rute yang dicek: {rute_indeks}")
    
    pelanggaran = 0
    
    # 1. CEK TITIK AWAL DAN AKHIR (Wajib 0)
    if rute_indeks[0] == 0 and rute_indeks[-1] == 0:
        print("  ✅ [PASS] Titik Awal dan Akhir adalah Depot (0).")
    else:
        print("  ❌ [FAIL] Rute tidak berawal atau berakhir di Depot!")
        pelanggaran += 1

    # 2. CEK DUPLIKASI (Puskesmas dikunjungi lebih dari 1 kali)
    # Potong rute tengah (tanpa depot awal dan akhir)
    rute_tengah = rute_indeks[1:-1]
    if len(rute_tengah) == len(set(rute_tengah)):
        print("  ✅ [PASS] Tidak ada Puskesmas yang dikunjungi dua kali.")
    else:
        print("  ❌ [FAIL] Ditemukan duplikasi titik yang dikunjungi!")
        pelanggaran += 1

    # 3. CEK TITIK TERLEWAT (Apakah semua Puskesmas didatangi?)
    titik_wajib = set(range(1, jumlah_titik_seharusnya))
    titik_dikunjungi = set(rute_tengah)
    titik_terlewat = titik_wajib - titik_dikunjungi
    
    if len(titik_terlewat) == 0:
        print("  ✅ [PASS] Semua Puskesmas di klaster ini berhasil dikunjungi.")
    else:
        print(f"  ❌ [FAIL] Ada titik yang terlewat: {titik_terlewat}")
        pelanggaran += 1
        
    # KESIMPULAN
    if pelanggaran == 0:
        print("  🌟 KESIMPULAN: RUTE 100% VALID DAN AMAN!\n")
    else:
        print(f"  ⚠️ KESIMPULAN: TERDAPAT {pelanggaran} PELANGGARAN CONSTRAINT!\n")

print("="*50)
for klaster, data in hasil_semua_klaster.items():
    # Mengambil matriks untuk tahu ada berapa titik seharusnya di klaster tersebut
    df_matriks = pd.read_csv(f"matriks_jarak_{klaster}.csv", index_col=0)
    jumlah_titik_asli = len(df_matriks)
    
    # Panggil fungsi cek
    cek_constraint_rute(klaster, data['rute_indeks'], jumlah_titik_asli)

🔍 MEMERIKSA CONSTRAINT KLASTER: BARAT
Rute yang dicek: [0, 11, 1, 12, 2, 9, 10, 4, 8, 7, 6, 5, 3, 0]
  ✅ [PASS] Titik Awal dan Akhir adalah Depot (0).
  ✅ [PASS] Tidak ada Puskesmas yang dikunjungi dua kali.
  ✅ [PASS] Semua Puskesmas di klaster ini berhasil dikunjungi.
  🌟 KESIMPULAN: RUTE 100% VALID DAN AMAN!

🔍 MEMERIKSA CONSTRAINT KLASTER: PUSAT
Rute yang dicek: [0, 1, 3, 8, 2, 5, 6, 7, 4, 0]
  ✅ [PASS] Titik Awal dan Akhir adalah Depot (0).
  ✅ [PASS] Tidak ada Puskesmas yang dikunjungi dua kali.
  ✅ [PASS] Semua Puskesmas di klaster ini berhasil dikunjungi.
  🌟 KESIMPULAN: RUTE 100% VALID DAN AMAN!

🔍 MEMERIKSA CONSTRAINT KLASTER: SELATAN
Rute yang dicek: [0, 13, 9, 5, 16, 10, 11, 12, 2, 3, 8, 15, 1, 7, 4, 14, 6, 0]
  ✅ [PASS] Titik Awal dan Akhir adalah Depot (0).
  ✅ [PASS] Tidak ada Puskesmas yang dikunjungi dua kali.
  ✅ [PASS] Semua Puskesmas di klaster ini berhasil dikunjungi.
  🌟 KESIMPULAN: RUTE 100% VALID DAN AMAN!

🔍 MEMERIKSA CONSTRAINT KLASTER: TIMUR
Rute yang dicek: 

In [15]:
import folium
import pandas as pd
import requests

# --- Fungsi Bantuan OSRM ---
def get_route_osrm(lat1, lon1, lat2, lon2):
    url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}?overview=full&geometries=geojson"
    try:
        response = requests.get(url).json()
        if response.get("code") == "Ok":
            return [[lat, lon] for lon, lat in response["routes"][0]["geometry"]["coordinates"]]
    except:
        pass
    return [[lat1, lon1], [lat2, lon2]]

# --- Setup Awal ---
klaster_list = ['Barat', 'Pusat', 'Selatan', 'Timur', 'Utara']
warna_klaster = {'Barat': 'red', 'Pusat': 'blue', 'Selatan': 'green', 'Timur': 'orange', 'Utara': 'purple'}
df_koordinat = pd.read_csv("koordinat_puskesmas.csv") 

m = folium.Map(location=[-7.3222876, 112.7717259], zoom_start=12, tiles='CartoDB positron')

# --- Looping Pembuatan Peta ---
for klaster in klaster_list:
    rute_nama = hasil_semua_klaster[klaster]['rute_nama']
    titik_klaster = []
    
    for nama in rute_nama:
        if "Gudang Farmasi" in nama:
            lat, lon = -7.3222876, 112.7717259
            folium.Marker(
                location=[lat, lon],
                popup="DEPOT: UPTD Gudang Farmasi",
                icon=folium.Icon(color='black', icon='home', prefix='fa') # Ikon rumah hitam
            ).add_to(m)
        else:
            row = df_koordinat[df_koordinat['Nama'] == nama].iloc[0]
            lat, lon = row['Latitude'], row['Longitude']
            
            # --- MARKER BIASA UNTUK PUSKESMAS ---
            nama_singkat = nama.replace("Puskesmas ", "").replace(" Surabaya", "")
            folium.CircleMarker(
                location=[lat, lon],
                radius=5,
                color=warna_klaster[klaster],
                fill=True,
                fill_opacity=0.7,
                tooltip=nama_singkat
            ).add_to(m)
            
        titik_klaster.append((lat, lon))

    # Gambar rute jalan raya asli
    for i in range(len(titik_klaster) - 1):
        lat1, lon1 = titik_klaster[i]
        lat2, lon2 = titik_klaster[i+1]
        jalan_asli = get_route_osrm(lat1, lon1, lat2, lon2)
        folium.PolyLine(jalan_asli, color=warna_klaster[klaster], weight=3, opacity=0.7).add_to(m)

m.save("hasil_rute_ver2.html")
m

In [16]:
import json

# 1. Siapkan wadah untuk format JSON yang baru
json_export = {}

# 2. Looping data hasil eksekusi dari variabel 'hasil_semua_klaster'
for klaster, data in hasil_semua_klaster.items():
    klaster_lower = klaster.lower() # Ubah "Barat" jadi "barat"
    
    json_export[klaster_lower] = {
        "rute_nama": data['rute_nama'],
        "jarak_optimasi": round(data['jarak'], 3), # Dibulatkan 3 angka di belakang koma biar rapi
        
        # Ambil riwayat konvergensi jika sudah kamu tambahkan di algoritma. 
        # Jika belum, otomatis akan diisi array kosong [] agar tidak error.
        "riwayat_konvergensi": data.get('riwayat_konvergensi', []) 
    }

# 3. Langsung tembak ke file di folder output_json yang sudah ada
path_simpan = 'rute_ga.json'

with open(path_simpan, 'w') as file:
    json.dump(json_export, file, indent=4)

print(f"✅ Mantap! Data JSON berhasil di-update dan disimpan di: {path_simpan}")

✅ Mantap! Data JSON berhasil di-update dan disimpan di: rute_ga.json
